In [ ]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, Dataloader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as pyplot


In [ ]:
torch.manual_seed(42)

In [ ]:
import pandas as pd

file_path = r"C:\Users\Harshu\Downloads\archive (3)\fashion-mnist_train.csv"

df = pd.read_csv(file_path)
print(df.head())

In [ ]:
# Create a 4*4  grid of images
fig, axes = plt.subplot(4,4 , figsize=(10,10))
fig.suptitle("First 16 images",fontsize = 16)

# plot the first 16 images from the dataset
for i , ax in enumerate(axes.flat):
    img = df.iloc[i,1:].values.reshape(28,28)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"Label:{df.iloc[i,0]}")

plt.tight_layout(rect = [0,0,1,0.96])
plt.show


In [ ]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):

        # Convert to pytorch tensors
        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1,1,28,28)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index] 
        

In [ ]:
train_dataset = CustomDataset(X_train, y_train)

In [ ]:
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
class MyNN(nn.Module):
    def __init__(self,input_features):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32 , kernel_size = 3, padding= 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride = 2),

            nn.Conv2d(32, 32 , kernel_size = 3, padding= 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride = 2),
        )


        self.classifier = nn.Sequential(
            nn.flatten()
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(p=0.4),


            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.4,)

            nn.Linear(64,10)
        )

    def forward(self , x):
        x = self.features(x)
        x = self.classifier(x)

        return x